# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammadfahadkhan-max/Week-01-ML-FlyRank-AI-/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
%pip install -q duckdb
import duckdb, os
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

In [26]:
# ---- Signal audit: build a click-growth label across dev + sealed test months ----

# 1. Aggregate March (dev) per (client, content): mean impressions, mean position
march = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_impressions) AS mar_impr,
           AVG(gsc_sum_position / NULLIF(gsc_impressions,0)) AS mar_pos,
           SUM(gsc_clicks) AS mar_clicks
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY 1,2
""").df()

# 2. Same aggregation for June (sealed test)
june = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS jun_clicks
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-06*/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY 1,2
""").df()

print("March rows:", march.shape[0], "| June rows:", june.shape[0])

# 3. Join, define growth label
merged = march.merge(june, on=['client_hash_id','content_hash_id'], how='inner')
merged['grew'] = (merged['jun_clicks'] > merged['mar_clicks']).astype(int)
print("Matched content pieces:", merged.shape[0])
print("Growth rate (label balance):", merged['grew'].mean())

# 4. Distributions (heavy tails check)
print(merged[['mar_impr','mar_pos','mar_clicks']].describe())

# 5. Three signal tests vs the label
for col in ['mar_impr','mar_pos','mar_clicks']:
    grew_mean = merged.loc[merged['grew']==1, col].mean()
    flat_mean = merged.loc[merged['grew']==0, col].mean()
    print(f"{col}: grew_mean={grew_mean:.3f} vs flat_mean={flat_mean:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 176738 | June rows: 208636
Matched content pieces: 137728
Growth rate (label balance): 0.16979844330855018
            mar_impr        mar_pos    mar_clicks
count  137728.000000  137728.000000  137728.00000
mean       68.768662      15.996112       5.90890
std       199.934467      16.616720      30.13703
min         1.000000       0.000000       0.00000
25%         3.600000       5.094403       0.00000
50%        13.800000       9.019767       0.00000
75%        56.322581      21.080022       3.00000
max     21280.137931     286.000000    5668.00000
mar_impr: grew_mean=85.938 vs flat_mean=65.257
mar_pos: grew_mean=13.265 vs flat_mean=16.555
mar_clicks: grew_mean=4.984 vs flat_mean=6.098


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

All three key fields are heavily right-skewed. mar_impr ranges from 1 to 21,280 with a mean of 68.8 but a median of only 13.8 — a small number of high-visibility pages pull the average way up. mar_clicks is even more skewed: median is 0, meaning most content gets zero clicks in a given month, while the max is 5,668. mar_pos ranges 0–286 with mean 16.0 and median 9.0, also right-skewed but less extreme. These heavy tails mean averages alone are misleading — any model or threshold built on these fields needs to account for the long tail, not just the mean.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(merged[['mar_impr','mar_pos','mar_clicks']].describe())

            mar_impr        mar_pos    mar_clicks
count  137728.000000  137728.000000  137728.00000
mean       68.768662      15.996112       5.90890
std       199.934467      16.616720      30.13703
min         1.000000       0.000000       0.00000
25%         3.600000       5.094403       0.00000
50%        13.800000       9.019767       0.00000
75%        56.322581      21.080022       3.00000
max     21280.137931     286.000000    5668.00000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal 1 — mar_impr (impressions): CONFIRMED. Growers averaged 85.9 March impressions vs 65.3 for non-growers — more visibility is associated with future click growth.

Signal 2 — mar_pos (average position): CONFIRMED. Growers averaged position 13.3 vs 16.6 for non-growers (lower = better) — better-ranked content is associated with future growth.

Signal 3 — mar_clicks (baseline clicks): MIXED. Growers had fewer March clicks on average (4.98) than non-growers (6.10) — the opposite of the naive expectation. Median mar_clicks is 0, so any page starting at zero only needs one click in June to count as "grew" — this is a floor effect in the label definition, not a trustworthy signal on its own.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for col in ['mar_impr','mar_pos','mar_clicks']:
    grew_mean = merged.loc[merged['grew']==1, col].mean()
    flat_mean = merged.loc[merged['grew']==0, col].mean()
    print(f"{col}: grew_mean={grew_mean:.3f} vs flat_mean={flat_mean:.3f}")

mar_impr: grew_mean=85.938 vs flat_mean=65.257
mar_pos: grew_mean=13.265 vs flat_mean=16.555
mar_clicks: grew_mean=4.984 vs flat_mean=6.098


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [29]:
import pandas as pd

In [30]:
import pandas as pd

merged['pos_bucket'] = pd.cut(merged['mar_pos'], bins=[-1,3,10,20,50,300],
                                labels=['1-3','4-10','11-20','21-50','50+'])
bucket_growth = merged.groupby('pos_bucket', observed=True)['grew'].agg(['mean','count'])
print(bucket_growth)

                mean  count
pos_bucket                 
1-3         0.180245  12250
4-10        0.175346  61604
11-20       0.181984  27420
21-50       0.180638  28261
50+         0.034298   8193


Flag tested: FlyRank's common assumption that "worse position = higher priority for review" — i.e. growth likelihood should decline steadily as position gets worse.

Verdict: FALSE (as a monotonic rule). Growth rate is essentially flat across positions 1–50 (17.5%–18.2% for buckets 1-3, 4-10, 11-20, and 21-50 — no meaningful difference between them). The assumption that "better position = more likely to grow" does NOT hold in this range; a page ranked 4-10 is just as likely to grow as one ranked 1-3. The one real signal is a cliff at position 50+, where growth rate drops to 3.4% — less than a fifth of every other bucket. So the flag's logic only holds at the extreme tail, not as a smooth gradient.

In [31]:
print(bucket_growth)

                mean  count
pos_bucket                 
1-3         0.180245  12250
4-10        0.175346  61604
11-20       0.181984  27420
21-50       0.180638  28261
50+         0.034298   8193


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team should not use position as a fine-grained priority score across the top 50 ranks — a page at position 5 and a page at position 45 have statistically the same chance of growing, so ranking review priority by position alone within that range wastes effort sorting pages that are functionally equivalent. The one place position is genuinely informative is as a coarse cutoff: pages beyond position 50 grow at roughly 1/5 the rate of everything else, so that threshold is worth using as a red flag, not the position value itself as a continuous priority score.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.